[![Ouvrir dans Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DEDJIABDOUL/omicaa-lab-machine-learning-bioinformatics/blob/main/notebooks/01_python_fondamentaux.ipynb)

# Notebook 01 : Python scientifique minimal pour la biologie

**Omicaa Lab | Machine Learning et Deep Learning pour les biologistes** | Auteur : DEDJI ABDOUL BATINOU

## Introduction scientifique

Un script d'analyse est une suite d'instructions qui transforme une donnée en résultat. Pour un biologiste, la première barrière n'est pas mathématique : c'est la lecture d'un script, la modification d'un paramètre et la correction d'une erreur. Ce notebook installe le vocabulaire minimal de Python en l'appliquant à un objet biologique concret, la séquence nucléotidique.

## Objectifs techniques

1. Manipuler variables, types scalaires, chaînes, listes et dictionnaires.
2. Écrire des conditions, des boucles et des fonctions avec arguments et valeur de retour.
3. Importer des modules, gérer une exception élémentaire et lire un fichier FASTA.
4. Lire une trace d'erreur (traceback) et corriger un script.

## Prérequis

* Aucun prérequis informatique.
* Un compte Google pour Colab, ou un environnement Python 3.10 ou supérieur.

## Résultat attendu

Une fonction de lecture FASTA et de calcul de contenu en GC, validée par des tests automatiques (assert), qui servira de modèle de rigueur pour toute la suite.

## 1. Contrôle de l'environnement

La première cellule d'un notebook vérifie les versions. Un résultat n'est interprétable que si l'on sait avec quel logiciel il a été obtenu.

In [1]:
import sys, platform
print("Python :", platform.python_version())
assert sys.version_info >= (3, 9), "Python 3.9 ou supérieur requis"
print("Environnement valide.")

Python : 3.12.3
Environnement valide.


## 2. Variables et types scalaires

Une **variable** est un nom qui désigne une valeur en mémoire. Le **type** de la valeur détermine les opérations possibles : on additionne deux nombres, on concatène deux chaînes, mais on ne peut pas additionner un nombre et une chaîne sans conversion.

| Type | Exemple | Usage biologique |
|---|---|---|
| `int` | `3200` | longueur d'une séquence |
| `float` | `0.47` | contenu en GC |
| `str` | `"ATGC"` | séquence, identifiant |
| `bool` | `True` | gène exprimé ou non |
| `NoneType` | `None` | valeur absente |

In [2]:
identifiant = "gene_demo_001"
longueur = 12
gc = 0.5833
exprime = True
annotation = None

for nom, valeur in [("identifiant", identifiant), ("longueur", longueur), ("gc", gc),
                    ("exprime", exprime), ("annotation", annotation)]:
    print(f"{nom:12s} -> {valeur!r:18} type : {type(valeur).__name__}")

# Les conversions de type sont explicites.
print(int("12") + 1, float("0.5") * 2, str(12) + " pb")

identifiant  -> 'gene_demo_001'    type : str
longueur     -> 12                 type : int
gc           -> 0.5833             type : float
exprime      -> True               type : bool
annotation   -> None               type : NoneType
13 1.0 12 pb


L'**f-string** (`f"...{variable}..."`) insère des valeurs dans un texte. Le suffixe `:.2f` fixe deux décimales. La partie après le deux-points est un format d'affichage, qui ne modifie pas la valeur.

## 3. Chaînes de caractères : une séquence est une chaîne

Une chaîne est une suite ordonnée de caractères. L'indexation commence à **0**. Le slicing `s[a:b]` extrait les positions de `a` inclus à `b` exclu.

In [3]:
seq = "ATGGCGTACGTTAGC"
print("Longueur       :", len(seq))
print("Premier codon  :", seq[0:3])
print("Dernier codon  :", seq[-3:])
print("Nombre de G    :", seq.count("G"))
print("Minuscules     :", seq.lower())
print("ARN            :", seq.replace("T", "U"))

complement = {"A": "T", "T": "A", "G": "C", "C": "G"}
brin_complementaire = "".join(complement[b] for b in seq)
print("Complémentaire :", brin_complementaire)

Longueur       : 15
Premier codon  : ATG
Dernier codon  : AGC
Nombre de G    : 5
Minuscules     : atggcgtacgttagc
ARN            : AUGGCGUACGUUAGC
Complémentaire : TACCGCATGCAATCG


## 4. Listes et dictionnaires

Une **liste** est une collection ordonnée et modifiable. Un **dictionnaire** associe une clé à une valeur : c'est la structure naturelle pour relier un identifiant de gène à une mesure.

In [4]:
codons = [seq[i:i+3] for i in range(0, len(seq) - len(seq) % 3, 3)]
print("Codons :", codons, "| nombre :", len(codons))
codons.append("TAA")
print("Après append :", codons[-2:])

expression = {"BRCA1": 5.2, "TP53": 7.9, "GAPDH": 12.4}
expression["MYC"] = 9.1
print("TP53 :", expression["TP53"])
print("EGFR (absent) :", expression.get("EGFR", "non mesuré"))
for gene, valeur in expression.items():
    print(f"{gene:6s} {valeur:5.1f}")

Codons : ['ATG', 'GCG', 'TAC', 'GTT', 'AGC'] | nombre : 5
Après append : ['AGC', 'TAA']
TP53 : 7.9
EGFR (absent) : non mesuré
BRCA1    5.2
TP53     7.9
GAPDH   12.4
MYC      9.1


La **compréhension de liste** `[expression for x in iterable]` est une écriture compacte d'une boucle qui construit une liste. Elle sera très utilisée en analyse de données.

## 5. Conditions et boucles

Une condition (`if`, `elif`, `else`) choisit une branche d'exécution. Une boucle (`for`, `while`) répète un bloc. **L'indentation** de quatre espaces définit ce qui appartient au bloc : en Python, elle fait partie de la syntaxe.

In [5]:
def classer(niveau):
    if niveau >= 10:
        return "forte"
    elif niveau >= 6:
        return "modérée"
    else:
        return "faible"

for gene, valeur in expression.items():
    print(f"{gene:6s} expression {classer(valeur)}")

comptes = {"A": 0, "C": 0, "G": 0, "T": 0}
for base in seq:
    comptes[base] += 1
print(comptes)
assert sum(comptes.values()) == len(seq)   # contrôle de cohérence

BRCA1  expression faible
TP53   expression modérée
GAPDH  expression forte
MYC    expression modérée
{'A': 3, 'C': 3, 'G': 5, 'T': 4}


## 6. Fonctions : arguments, valeur de retour, documentation

Une **fonction** encapsule une opération réutilisable. Les **arguments** sont ses entrées, la **valeur de retour** sa sortie. La docstring explique le contrat de la fonction. Une fonction bien écrite lève une erreur explicite quand son entrée est invalide, au lieu de produire un résultat faux en silence.

In [6]:
def contenu_gc(sequence: str) -> float:
    """Retourne la proportion de bases G ou C dans une séquence d'ADN.

    Lève ValueError si la séquence est vide ou contient une base inattendue.
    """
    sequence = sequence.upper()
    if len(sequence) == 0:
        raise ValueError("séquence vide")
    invalides = set(sequence) - set("ACGTN")
    if invalides:
        raise ValueError(f"bases inattendues : {sorted(invalides)}")
    return (sequence.count("G") + sequence.count("C")) / len(sequence)

print(round(contenu_gc(seq), 4))
assert contenu_gc("GGCC") == 1.0
assert contenu_gc("ATAT") == 0.0
assert abs(contenu_gc("ATGC") - 0.5) < 1e-12
print("Tests de contenu_gc réussis.")

0.5333
Tests de contenu_gc réussis.


## 7. Imports et exceptions

Un **module** regroupe des fonctions déjà écrites. `import math` rend ses outils disponibles sous le préfixe `math.`. Une **exception** signale une situation anormale. Le bloc `try/except` permet de la traiter sans interrompre tout le script.

In [7]:
import math

print("log2(8)  =", math.log2(8))
print("racine(2)=", round(math.sqrt(2), 4))

for candidate in ["ATGC", "ATXC", ""]:
    try:
        print(repr(candidate), "->", contenu_gc(candidate))
    except ValueError as erreur:
        print(repr(candidate), "-> entrée refusée :", erreur)

log2(8)  = 3.0
racine(2)= 1.4142
'ATGC' -> 0.5
'ATXC' -> entrée refusée : bases inattendues : ['X']
'' -> entrée refusée : séquence vide


## 8. Lire un fichier FASTA

Le format FASTA alterne une ligne d'en-tête commençant par `>` et une ou plusieurs lignes de séquence. Le notebook crée d'abord un petit fichier d'exemple, puis le relit avec une fonction qui valide sa structure (Étape B : validation des entrées).

In [8]:
from pathlib import Path

dossier = Path("donnees_demo")
dossier.mkdir(exist_ok=True)
fichier = dossier / "exemple.fasta"
fichier.write_text(">seq1 gene fictif A\nATGGCGTACG\nTTAGC\n>seq2 gene fictif B\nGGGCCCATAT\n>seq3 gene fictif C\nATATATATAT\n")

def lire_fasta(chemin) -> dict:
    """Lit un fichier FASTA et retourne {identifiant: séquence}."""
    chemin = Path(chemin)
    if not chemin.exists():
        raise FileNotFoundError(f"fichier introuvable : {chemin}")
    sequences, courant = {}, None
    for ligne in chemin.read_text().splitlines():
        ligne = ligne.strip()
        if not ligne:
            continue
        if ligne.startswith(">"):
            courant = ligne[1:].split()[0]
            if courant in sequences:
                raise ValueError(f"identifiant dupliqué : {courant}")
            sequences[courant] = ""
        else:
            if courant is None:
                raise ValueError("séquence sans en-tête")
            sequences[courant] += ligne
    return sequences

fasta = lire_fasta(fichier)
for identifiant, s in fasta.items():
    print(f"{identifiant}  longueur={len(s):2d}  GC={contenu_gc(s):.2f}")

assert list(fasta) == ["seq1", "seq2", "seq3"]
assert len(fasta["seq1"]) == 15

seq1  longueur=15  GC=0.53
seq2  longueur=10  GC=0.60
seq3  longueur=10  GC=0.00


## 9. Lire une trace d'erreur

Une erreur n'est pas un échec : c'est une information. La **traceback** se lit de bas en haut. La dernière ligne donne le type et le message de l'erreur, les lignes précédentes indiquent l'enchaînement des appels. Le code suivant provoque volontairement une erreur, la capture et l'affiche.

In [9]:
import traceback

def moyenne(valeurs):
    return sum(valeurs) / len(valeurs)

try:
    moyenne([])
except ZeroDivisionError:
    print(traceback.format_exc())

Traceback (most recent call last):
  File "/tmp/ipykernel_13847/2320861215.py", line 7, in <module>
    moyenne([])
  File "/tmp/ipykernel_13847/2320861215.py", line 4, in moyenne
    return sum(valeurs) / len(valeurs)
           ~~~~~~~~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero



Cause : la liste est vide, donc `len(valeurs)` vaut 0. Les erreurs courantes d'un débutant sont `NameError` (nom non défini, souvent une cellule non exécutée), `TypeError` (opération sur des types incompatibles), `KeyError` (clé absente d'un dictionnaire), `IndexError` (position hors limites) et `FileNotFoundError` (mauvais chemin).

## 10. Exercices progressifs

Chaque exercice est suivi d'une solution. Essayez d'abord de les résoudre dans une cellule vide avant d'ouvrir la solution.

**Exercice 1.** Calculez la longueur de la séquence `"ATGAAATAG"` et son nombre de codons.

*Solution de l'exercice 1*

In [10]:
s = "ATGAAATAG"
print(len(s), len(s) // 3)
assert (len(s), len(s) // 3) == (9, 3)

9 3


**Exercice 2.** Écrivez une fonction `complement_inverse(sequence)` qui retourne le brin complémentaire inverse (lu de 5' vers 3'). Testez-la sur `"ATGC"`, attendu : `"GCAT"`.

*Solution de l'exercice 2*

In [11]:
def complement_inverse(sequence):
    table = {"A": "T", "T": "A", "G": "C", "C": "G"}
    return "".join(table[b] for b in reversed(sequence.upper()))

assert complement_inverse("ATGC") == "GCAT"
assert complement_inverse(complement_inverse("ATGGCGTAC")) == "ATGGCGTAC"
print(complement_inverse("ATGC"))

GCAT


**Exercice 3.** À partir du dictionnaire `expression`, construisez la liste des gènes dont l'expression dépasse 7, triée par ordre alphabétique.

*Solution de l'exercice 3*

In [12]:
forts = sorted(g for g, v in expression.items() if v > 7)
print(forts)
assert forts == ["GAPDH", "MYC", "TP53"]

['GAPDH', 'MYC', 'TP53']


**Exercice 4.** Écrivez `compter_codon(sequence, codon)` qui compte les occurrences d'un codon dans le cadre de lecture 0 (positions 0, 3, 6, ...). Vérifiez que `compter_codon("ATGATGAAA", "ATG")` vaut 2.

*Solution de l'exercice 4*

In [13]:
def compter_codon(sequence, codon):
    codons = [sequence[i:i+3] for i in range(0, len(sequence) - 2, 3)]
    return codons.count(codon)

assert compter_codon("ATGATGAAA", "ATG") == 2
print(compter_codon("ATGATGAAA", "ATG"))

2


## 11. Synthèse des compétences acquises

* Distinguer les types `int`, `float`, `str`, `bool` et `None`, et convertir explicitement.
* Utiliser le slicing, les listes, les dictionnaires et les compréhensions.
* Écrire une fonction documentée qui valide ses entrées et la protéger par des `assert`.
* Lire un fichier structuré en vérifiant sa cohérence.
* Lire une traceback et identifier la cause d'une erreur.

**Suite.** Le notebook 02 introduit NumPy et pandas, qui remplacent les boucles explicites par des opérations sur des tableaux entiers.